In [ ]:
import os, shutil, subprocess, json, pickle, warnings, logging, optuna
from functools import partial
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt, seaborn as sns
import matplotlib.dates as mdates
from scipy.stats import qmc
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, RationalQuadratic, WhiteKernel, ConstantKernel as C
)
from optuna.importance import MeanDecreaseImpurityImportanceEvaluator
from sklearn.ensemble import RandomForestRegressor
warnings.filterwarnings('ignore', category=UserWarning)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Setup functions
def generate_lhs_samples(parameters_range, n_samples, seed=42):
    param_names = list(parameters_range.keys())
    sampler = qmc.LatinHypercube(d=len(param_names), seed=seed, optimization='random-cd')
    sample_lhs = sampler.random(n=n_samples)
    lower_bounds = [parameters_range[p][0] for p in param_names]
    upper_bounds = [parameters_range[p][1] for p in param_names]
    sample_scaled = qmc.scale(sample_lhs, lower_bounds, upper_bounds)
    return pd.DataFrame(sample_scaled, columns=param_names, index=np.arange(1, n_samples + 1))

def modify_mdu_key(mdu_lines: list, key: str, value: str = '') -> list:
    mdu = mdu_lines.copy()
    index, line = next((i, line) for i, line in enumerate(mdu) if line.strip().startswith(key))
    new = line.split('=')
    new1 = new[1].split('#')
    mdu[index] = f'{new[0]}= {value.ljust(len(new1[0])-2)} #{new1[1]}'
    return mdu

# Remove outliers from the measured data using rolling window method
def remove_rolling_outliers(df: pd.DataFrame, column: str, window: int = 20, n_std: float = 3.0) -> pd.DataFrame:
    df_clean = df.copy()
    rolling_mean = df_clean[column].rolling(window=window, center=True, min_periods=3).mean()
    rolling_std = df_clean[column].rolling(window=window, center=True, min_periods=3).std()
    lower = rolling_mean - n_std * rolling_std
    upper = rolling_mean + n_std * rolling_std
    mask = (df_clean[column] >= lower) & (df_clean[column] <= upper) | df_clean[column].isna()
    return df_clean[mask].copy()

def compute_weighted_rmse(depths: list, rmse_list: list, method: str = 'equal') -> float:
    depth_arr, rmse_arr = np.array(depths), np.array(rmse_list)
    valid = np.isfinite(depth_arr) & np.isfinite(rmse_arr) & (rmse_arr >= 0)
    if not valid.any(): return np.nan
    v_depths, v_rmse = depth_arr[valid], rmse_arr[valid]
    if method == 'surface': weights = 1.0 / v_depths
    elif method == 'deep': weights = v_depths
    elif method == 'equal': weights = np.ones(len(v_depths))
    weights = weights / weights.sum()
    score = np.sqrt(np.sum(weights * v_rmse ** 2))
    return score

def interpolate_profile_to_depths(group, depths):
    g = group.sort_values("depth").drop_duplicates("depth").reset_index(drop=True)
    d_arr, t_arr = g["depth"].to_numpy(), g["temperature"].to_numpy()
    if len(d_arr) < 2: return pd.DataFrame()
    t_start = g["TIMESTAMP"].iloc[0]
    dt_sec = (g["TIMESTAMP"] - t_start).dt.total_seconds().to_numpy()
    rows = []
    for d in depths:
        if d_arr.min() <= d <= d_arr.max():
            temp_interp = float(np.interp(d, d_arr, t_arr))
            el_sec = float(np.interp(d, d_arr, dt_sec))
            ts = t_start + pd.to_timedelta(el_sec, unit="s").round("1s")
            row = {"TIMESTAMP": ts}
            for depth_col in depths:
                row[f"obs_{depth_col}"] = temp_interp if depth_col == d else np.nan
            rows.append(row)
    return pd.DataFrame(rows)

def run(mdu_path, path, dir):
    bat_path = os.path.normpath(os.path.join(dir, r"backend\softs\x64\dflowfm\scripts\run_dflowfm.bat"))
    new_path = os.path.normpath(os.path.join(dir, mdu_path))
    command = ["cmd.exe", "/c", bat_path, "--autostartstop", new_path]
    print("=" * 80)
    print(f"[STARTING] {mdu_path}")
    print("=" * 80)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1, cwd=path)
    error_detected = False
    # Stream logs
    for line in process.stdout:
        line = line.strip()
        if not line: continue
        print(line)
        # Catch error messages
        if "forrtl:" in line.lower() or "error" in line.lower(): error_detected = True
    return_code = process.wait()
    print("=" * 80)
    if return_code == 0 and not error_detected:
        print(f"[COMPLETED] {mdu_path}")
        print("=" * 80)
        return True
    else:
        print(f"[FAILED] {mdu_path}, exit code={return_code}")
        print("=" * 80)
        return False

def read_and_interpolate_his(case_output_dir, station_name, depths):
    his_file = os.path.join(case_output_dir, 'output', 'FlowFM_his.nc')
    if not os.path.exists(his_file): return pd.DataFrame()
    with xr.open_dataset(his_file) as ds:
        names = [n.decode('utf-8').strip() if isinstance(n, bytes) else str(n).strip() for n in ds['station_name'].values]
        if station_name not in names: return pd.DataFrame()
        st_idx = names.index(station_name)
        t_his = ds['temperature'].isel(stations=st_idx).values
        z_his = ds['zcoordinate_c'].isel(stations=st_idx).values
        wl_his = ds['waterlevel'].isel(stations=st_idx).values
        sim_times = pd.to_datetime(ds['time'].values)
    n_steps = len(sim_times)
    sim_interp = np.full((n_steps, len(depths)), np.nan)
    for i in range(n_steps):
        t_row, z_row, wl = t_his[i, :], z_his[i, :], wl_his[i]
        tgt_elev = wl - np.asarray(depths)
        mask = np.isfinite(t_row) & np.isfinite(z_row)
        if mask.sum() >= 2:
            z_valid, t_valid = z_row[mask], t_row[mask]
            order = np.argsort(z_valid)
            z_sorted, t_sorted = z_valid[order], t_valid[order]
            in_bounds = (tgt_elev >= z_sorted.min()) & (tgt_elev <= z_sorted.max())
            sim_interp[i, in_bounds] = np.interp(tgt_elev[in_bounds], z_sorted, t_sorted)
    df_sim = pd.DataFrame(sim_interp, columns=[f"sim_{d}" for d in depths], index=sim_times).reset_index().rename(columns={"index": "TIMESTAMP"})
    return df_sim

## Read and prepare observation

In [ ]:
measured_path = r'Calibration\Profiler_modem_PFL_Step.dat'
depths_selected = [5, 10, 30, 40, 50]
measured_df = pd.read_csv(measured_path, delimiter=',', header=0, skiprows=[0, 2, 3], low_memory=False)
measured_df["TIMESTAMP"] = pd.to_datetime(measured_df["TIMESTAMP"])
measured_df["temperature"] = pd.to_numeric(measured_df["sensorParms(1)"], errors='coerce')
measured_df["depth"] = pd.to_numeric(measured_df["sensorParms(9)"], errors='coerce')
measured_df = measured_df[["TIMESTAMP", "temperature", "depth"]].dropna(subset=["TIMESTAMP"])
# Get data for the year 2024
df_2024 = measured_df[measured_df["TIMESTAMP"].dt.year == 2024].sort_values("TIMESTAMP").reset_index(drop=True)
df_2024["temperature"] = df_2024["temperature"].interpolate(method='linear', limit_direction='both')
df_2024["depth"] = df_2024["depth"].interpolate(method='linear', limit_direction='both')
# Detect measurement periods
df_2024['new_profile'] = ((df_2024["depth"] < 1.5) & (df_2024["depth"].shift(1) > 5.0))
df_2024['profile_id'] = df_2024['new_profile'].cumsum()
temp_measured = (
    df_2024.groupby("profile_id", group_keys=False)
    .apply(lambda g: interpolate_profile_to_depths(g, depths_selected))
    .reset_index(drop=True).sort_values("TIMESTAMP").reset_index(drop=True)
)

## Initial parameters

In [ ]:
mdu_template_path = r'Calibration\3_MonthBest\dflowfm\FlowFM.mdu'
base_workspace = r'C:\Users\vanln\Downloads\Hydro-AI-Platform'
test_folder = r'Calibration\test'
param_samples_file = os.path.join(test_folder, 'parameter_samples.csv')
os.makedirs(test_folder, exist_ok=True)
parameters_range = {
    'Secchidepth': [1.0, 20.0], 'Dicoww': [0, 1e-3], 
    'Vicoww': [0, 5e-3], 'Vicouv': [0.1, 1.5], 'Dicouv': [0.1, 1.5], 
}

In [ ]:
# Create Latin Hypercube Sampling (LHS) design for the parameters
n_samples = 50
df_params = generate_lhs_samples(parameters_range=parameters_range, n_samples=n_samples)
df_params.to_csv(param_samples_file)

## Create sample datasets

In [ ]:
# Read mdu file
with open(mdu_template_path, 'r', encoding='utf-8', errors='ignore') as f:
    mdu_base = f.readlines()
his_interval = 3600 * 6
mdu_base = modify_mdu_key(mdu_base, 'WaqInterval', '0')
mdu_base = modify_mdu_key(mdu_base, 'RstInterval', '0')
mdu_base = modify_mdu_key(mdu_base, 'RestartFile', '')
mdu_base = modify_mdu_key(mdu_base, 'HisInterval', str(his_interval))
mdu = modify_mdu_key(mdu_base, 'Kmx', '40')
# new_line = 'verticalAdvectionType             = 4               #\n'
# new_line_normalized = new_line.strip().lower()
# if not any(line.strip().lower() == new_line_normalized for line in mdu):
#     for i, line in enumerate(mdu):
#         if line.strip().lower() == '[physics]':
#             mdu.insert(i + 1, new_line)
#             break
# Optimize his parameters
vars_to_keep = {'Wrihis_temperature', 'Wrihis_waterlevel_s1'}
for line in mdu_base:
    if line.strip().startswith(('Wrihis_', 'Wrimap_')) and '=' in line:
        var_name = line.split('=')[0].strip()
        val = '1' if var_name in vars_to_keep else '0'
        mdu_base = modify_mdu_key(mdu_base, var_name, val)

In [ ]:
# Create different scenarios
scenarios_dir = os.path.join(test_folder, 'scenarios')
os.makedirs(scenarios_dir, exist_ok=True)
parent_mdu_dir = os.path.dirname(mdu_template_path)
common_files = [f for f in os.listdir(parent_mdu_dir) if os.path.isfile(os.path.join(parent_mdu_dir, f)) and not f.endswith('.mdu')]
custom_meteo_params = {'CloudFactor', 'CloudOffset', 'AirTemperature'}

In [ ]:
# Store different scenarios
for case_id, row in df_params.iterrows():
    case_dir = os.path.join(scenarios_dir, str(case_id))
    os.makedirs(case_dir, exist_ok=True)
    # Copy files
    for f in common_files:
        shutil.copy(os.path.join(parent_mdu_dir, f), os.path.join(case_dir, f))
    # Update MDU
    mdu_case = mdu_base.copy()
    mdu_case = modify_mdu_key(mdu_case, 'MapInterval', str(his_interval) if case_id == 1 else '0')
    for p_name, p_val in row.items():
        if p_name not in custom_meteo_params:
            val_str = f"{p_val:.5e}" if isinstance(p_val, float) else str(p_val)
            mdu_case = modify_mdu_key(mdu_case, p_name, val_str)
    with open(os.path.join(case_dir, 'FlowFM.mdu'), 'w', encoding='utf-8') as f:
        f.writelines(mdu_case)
    # # Update file FlowFM_meteo.tim
    # meteo_src = os.path.join(parent_mdu_dir, 'FlowFM_meteo.tim')
    # if os.path.exists(meteo_src):
    #     meteo_data = np.loadtxt(meteo_src)
    #     # Update cloud cover (%)
    #     meteo_data[:, 3] = np.clip(meteo_data[:, 3] * row['CloudFactor'] + row['CloudOffset'], 0.0, 100.0)
    #     # Update air temperature
    #     meteo_data[:, 2] = meteo_data[:, 2] + row['AirTemperature']
    #     np.savetxt(os.path.join(case_dir, 'FlowFM_meteo.tim'), meteo_data, fmt='%.7e')

## Run cases

In [ ]:
error_cases = []
list_sorted = sorted(
    [int(x) for x in os.listdir(scenarios_dir) 
     if os.path.isdir(os.path.join(scenarios_dir, x))]
)
for case_id in list_sorted:
    c_dir = os.path.join(scenarios_dir, str(case_id))
    print(f"-> Running model: {case_id}/{n_samples}...")
    model_dir = os.path.join(test_folder, 'scenarios', str(case_id))
    model_path = os.path.join(model_dir, 'FlowFM.mdu')
    success = run(model_path, model_dir, base_workspace)
    if not success:
        print(f"Scenario {case_id} failed!")
        error_cases.append(case_id)
print(f'Number of error models: {len(error_cases)}')

## Prepare outputs

In [ ]:
summary_list, obs_station, rmse_method = [], 'Profiler', 'balanced'
df_params = pd.read_csv(param_samples_file, index_col=0)
for case_id in sorted([int(x) for x in os.listdir(scenarios_dir) if os.path.isdir(os.path.join(scenarios_dir, x))]):
    c_dir = os.path.join(scenarios_dir, str(case_id))
    df_sim = read_and_interpolate_his(c_dir, obs_station, depths_selected)
    if df_sim.empty: continue
    t_min = max(temp_measured["TIMESTAMP"].min(), df_sim["TIMESTAMP"].min())
    t_max = min(temp_measured["TIMESTAMP"].max(), df_sim["TIMESTAMP"].max())
    obs_sub = temp_measured[(temp_measured["TIMESTAMP"] >= t_min) & (temp_measured["TIMESTAMP"] <= t_max)]
    sim_sub = df_sim[(df_sim["TIMESTAMP"] >= t_min) & (df_sim["TIMESTAMP"] <= t_max)].set_index("TIMESTAMP")
    rmse_per_depth = []
    for d in depths_selected:
        obs_col, sim_col = f"obs_{d}", f"sim_{d}"
        obs_series = obs_sub[["TIMESTAMP", obs_col]].dropna().set_index("TIMESTAMP").sort_index()
        obs_series = obs_series[~obs_series.index.duplicated(keep='first')]
        obs_clean = remove_rolling_outliers(obs_series, obs_col)
        comb = sim_sub[[sim_col]].reindex(sim_sub.index.union(obs_clean.index)).sort_index()
        comb[sim_col] = comb[sim_col].interpolate(method="time")
        aligned = comb.loc[obs_clean.index, [sim_col]].join(obs_clean).dropna()
        if len(aligned) > 0:
            rmse_d = np.sqrt(mean_squared_error(aligned[obs_col], aligned[sim_col]))
            rmse_per_depth.append(rmse_d)
        else: rmse_per_depth.append(np.nan)
    weighted_rmse = compute_weighted_rmse(depths_selected, rmse_per_depth, rmse_method)
    row_params = df_params.loc[case_id].to_dict()
    row_params['RMSE'] = weighted_rmse
    summary_list.append(row_params)
summary_df = pd.DataFrame(summary_list).dropna(subset=['RMSE'])
summary_csv = os.path.join(test_folder, 'calibration_summary.csv')
summary_df.to_csv(summary_csv, index=False)
print(f"Đã tổng hợp kết quả 60 kịch bản:")

## Correlation analysis

In [ ]:
# Correlation
correlation = summary_df.corr()['RMSE'].sort_values(ascending=False)
print(correlation)

plt.figure(figsize=(12, 8))
sns.heatmap(summary_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(summary_df['RMSE'], bins=10, edgecolor='black', alpha=0.7)
plt.xlabel('RMSE (°C)')
plt.ylabel('Number of iterration')
plt.title('MSE Distribution')
plt.grid(True, alpha=0.3)
plt.subplot(1, 2, 2)
plt.scatter(range(len(summary_df)), summary_df['RMSE'], alpha=0.7)
plt.xlabel('Run')
plt.ylabel('RMSE (°C)')
plt.title('RMSE')
plt.grid(True, alpha=0.3)
plt.show()

## Build Surrogate model

In [ ]:
X, y = summary_df.drop(columns=['RMSE']), summary_df[['RMSE']]
# Chuẩn hóa dữ liệu
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y).ravel()
# Danh sách Kernel đánh giá
kernels_to_test = [
    (C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-3, 1e2)), 'RBF'),
    (C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2)) + WhiteKernel(1e-3, (1e-10, 1e-1)), 'RBF + White'),
    (C(1.0, (1e-3, 1e3)) * Matern(1.0, nu=1.5, length_scale_bounds=(1e-3, 1e2)), 'Matern_1.5'),
    (C(1.0, (1e-3, 1e3)) * Matern(1.0, nu=2.5, length_scale_bounds=(1e-3, 1e2)), 'Matern_2.5'),
    (C(1.0, (1e-3, 1e3)) * RationalQuadratic(1.0, 1.0, length_scale_bounds=(1e-9, 1e6)), 'RationalQuadratic')
]
cv_results = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for kernel_obj, k_name in kernels_to_test:
    gpr = GaussianProcessRegressor(kernel=kernel_obj, n_restarts_optimizer=30, alpha=1e-6, normalize_y=True, random_state=42)
    rmse_folds, r2_folds = [], []
    for train_idx, val_idx in kf.split(X_scaled):
        gpr_fold = clone(gpr)
        gpr_fold.fit(X_scaled[train_idx], y_scaled[train_idx])
        y_pred = gpr_fold.predict(X_scaled[val_idx])
        rmse_folds.append(np.sqrt(mean_squared_error(y_scaled[val_idx], y_pred)))
        r2_folds.append(r2_score(y_scaled[val_idx], y_pred))
    cv_results.append({
        'name': k_name, 'kernel': kernel_obj, 'rmse_mean': np.mean(rmse_folds),
        'rmse_std': np.std(rmse_folds), 'r2_mean': np.mean(r2_folds)
    })
best_cv = min(cv_results, key=lambda x: x['rmse_mean'])
print(f"Kernel tốt nhất: {best_cv['name']} (RMSE CV: {best_cv['rmse_mean']:.4f} ± {best_cv['rmse_std']:.4f})")

## Train and save model

In [ ]:
final_gpr = GaussianProcessRegressor(
    kernel=best_cv['kernel'], n_restarts_optimizer=50, 
    alpha=1e-6, normalize_y=True, random_state=42
)
final_gpr.fit(X_scaled, y_scaled)
# Lưu Model và Scaler
model_dir = os.path.join(test_folder, 'model')
os.makedirs(model_dir, exist_ok=True)
objects = [
    ('scaler_X', X_scaled), 
    ('scaler_y', y_scaled),
    ('gpr_model', final_gpr)
]
for scaler, data in objects:
    with open(os.path.join(model_dir, f'{scaler}.pkl'), "wb") as f:
        pickle.dump(data, f)
print(f"Đã lưu mô hình Gaussian Process tại: {model_dir}")

## Run Optuna

In [ ]:
# Run optimization with Optuna
def optuna_objective(trial, scaler_x, scaler_y_obj, model, p_ranges):
    trial_params = {}
    for p, bounds in p_ranges.items():
        # log_scale = True if 'ww' in p.lower() else False
        trial_params[p] = trial.suggest_float(p, bounds[0], bounds[1])
    df_in = pd.DataFrame([trial_params], columns=list(p_ranges.keys()))
    x_sc = scaler_x.transform(df_in)
    pred_sc = model.predict(x_sc)
    pred_orig = scaler_y_obj.inverse_transform(pred_sc.reshape(-1, 1)).ravel()
    return pred_orig[0]
optuna_func = partial(
    optuna_objective, scaler_x=scaler_X, scaler_y_obj=scaler_y, 
    model=final_gpr, p_ranges=parameters_range
)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    sampler=optuna.samplers.TPESampler(seed=42), direction='minimize', 
    study_name='delft3d_fm_calibration'
)
study.optimize(optuna_func, n_trials=2000, show_progress_bar=True)
print(f"Giá trị RMSE nhỏ nhất dự đoán: {study.best_value:.4f} °C")
print(f"\nBộ tham số tối ưu khuyến nghị:")
for param_k, param_v in study.best_params.items():
    if isinstance(param_v, float) and param_v < 0.001:
        print(f"  {param_k:<20}: {param_v:.6e}")
    else: print(f"  {param_k:<20}: {param_v:.6f}")

In [ ]:
# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df_trials = study.trials_dataframe()
ax = axes[0, 0]
cum_min = df_trials['value'].cummin()
ax.plot(cum_min, 'b-', lw=2, label='Best RMSE so far')
ax.set_xlabel('Iterations (Trials)', fontsize=11)
ax.set_ylabel('Smallest RMSE (°C)', fontsize=11)
ax.set_title("Optuna's Convergence Process", fontsize=13, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend()
ax = axes[0, 1]
ax.hist(df_trials['value'].dropna(), bins=25, alpha=0.7, color='royalblue', edgecolor='black')
ax.axvline(study.best_value, color='red', linestyle='--', lw=2, label=f'Best: {study.best_value:.4f} °C')
ax.set_xlabel('RMSE (°C)', fontsize=11)
ax.set_ylabel('Number of Trials', fontsize=11)
ax.set_title('RMSE Distribution Across Trials', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)
ax = axes[1, 0]
try:
    print('Compute feature importance from Optuna')
    importance = optuna.importance.get_param_importances(
        study, evaluator=MeanDecreaseImpurityImportanceEvaluator()
    )
except Exception:
    # Fallback: Tự tính feature importance bằng Random Forest từ Scikit-Learn
    print('Compute feature importance from Random Forest')
    param_cols = [c for c in df_trials.columns if c.startswith('params_')]
    X_tr = df_trials[param_cols].fillna(df_trials[param_cols].mean())
    y_tr = df_trials['value'].fillna(df_trials['value'].max())
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_tr, y_tr)
    clean_names = [c.replace('params_', '') for c in param_cols]
    importance = dict(zip(clean_names, rf.feature_importances_))

sorted_importance = dict(sorted(importance.items(), key=lambda item: item[1]))
param_names = list(sorted_importance.keys())
param_vals = list(sorted_importance.values())
colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(param_names)))
bars = ax.barh(param_names, param_vals, color=colors, edgecolor='black', alpha=0.85)
ax.set_xlabel('Relative Importance', fontsize=11)
ax.set_title('Importance of Parameters', fontsize=13, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.6, axis='x')
# Hiển thị phần trăm giá trị cạnh mỗi thanh
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.1%}', 
            va='center', ha='left', fontsize=9, color='black')
ax = axes[1, 1]
# Ưu tiên chọn tham số có độ quan trọng cao nhất, hoặc mặc định là 'Secchidepth'
most_important_var = param_names[-1] if len(param_names) > 0 else 'Secchidepth'
var = most_important_var if f'params_{most_important_var}' in df_trials.columns else 'Secchidepth'
if f'params_{var}' in df_trials.columns:
    sc = ax.scatter(df_trials[f'params_{var}'], df_trials['value'], alpha=0.6, s=15, c=df_trials['value'], cmap='coolwarm')
    best_val_param = study.best_params[var]
    if abs(best_val_param) < 0.001: label_text = f"Best: {best_val_param:.4e}"
    else: label_text = f"Best: {best_val_param:.3f}"
    ax.axvline(best_val_param, color='red', linestyle='--', lw=2, label=label_text)
    ax.set_xlabel(f'{var}', fontsize=11)
    ax.set_ylabel('RMSE (°C)', fontsize=11)
    ax.set_title(f'Sensitivity Analysis: {var}', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## Update new parameters and rerun Delft3D FM 

In [ ]:
final_dir = os.path.join(test_folder, 'model', 'final')
if os.path.exists(final_dir):
    shutil.rmtree(final_dir)
os.makedirs(final_dir, exist_ok=True)
# Copy toàn bộ file cấu hình cơ sở (lưới, địa hình, biên...)
for f in common_files:
    shutil.copy(os.path.join(parent_mdu_dir, f), os.path.join(final_dir, f))
# 2. Cập nhật các tham số tối ưu vào file FlowFM.mdu
mdu_final = mdu_base.copy()
# Cho phép xuất MAP file cho kịch bản tối ưu để xem trực quan hóa 2D/3D
mdu_final = modify_mdu_key(mdu_final, 'MapInterval', str(his_interval))
for param_name, param_val in study.best_params.items():
    if param_name not in custom_meteo_params:
        val_str = f"{param_val:.6e}" if isinstance(param_val, float) and abs(param_val) < 0.01 else f"{param_val:.6f}"
        mdu_final = modify_mdu_key(mdu_final, param_name, val_str)
final_mdu_path = os.path.join(final_dir, 'FlowFM.mdu')
with open(final_mdu_path, 'w', encoding='utf-8') as f:
    f.writelines(mdu_final)
# # 3. Cập nhật các tham số khí tượng vào FlowFM_meteo.tim (nếu có)
# meteo_src = os.path.join(parent_mdu_dir, 'FlowFM_meteo.tim')
# if os.path.exists(meteo_src):
#     meteo_data = np.loadtxt(meteo_src)
#     if 'CloudFactor' in study.best_params and 'CloudOffset' in study.best_params:
#         meteo_data[:, 3] = np.clip(
#             meteo_data[:, 3] * study.best_params['CloudFactor'] + study.best_params['CloudOffset'], 
#             0.0, 100.0
#         )
#     if 'AirTemperature' in study.best_params:
#         meteo_data[:, 2] = meteo_data[:, 2] + study.best_params['AirTemperature']
#     np.savetxt(os.path.join(final_dir, 'FlowFM_meteo.tim'), meteo_data, fmt='%.7e')
# Run a simulation
model_path = os.path.join(final_dir, 'FlowFM.mdu')
run_success = run(model_path, final_dir, base_workspace)
print(run_success)

In [ ]:
df_sim_final = read_and_interpolate_his(final_dir, obs_station, depths_selected)
if not df_sim_final.empty:
    t_min = max(temp_measured["TIMESTAMP"].min(), df_sim_final["TIMESTAMP"].min())
    t_max = min(temp_measured["TIMESTAMP"].max(), df_sim_final["TIMESTAMP"].max())
    obs_sub = temp_measured[(temp_measured["TIMESTAMP"] >= t_min) & (temp_measured["TIMESTAMP"] <= t_max)]
    sim_sub = df_sim_final[(df_sim_final["TIMESTAMP"] >= t_min) & (df_sim_final["TIMESTAMP"] <= t_max)].set_index("TIMESTAMP")
    performance_metrics = []
    aligned_data_dict = {}
    for d in depths_selected:
        obs_col, sim_col = f"obs_{d}", f"sim_{d}"
        obs_series = obs_sub[["TIMESTAMP", obs_col]].dropna().set_index("TIMESTAMP").sort_index()
        obs_series = obs_series[~obs_series.index.duplicated(keep='first')]
        obs_clean = remove_rolling_outliers(obs_series, obs_col)
        comb = sim_sub[[sim_col]].reindex(sim_sub.index.union(obs_clean.index)).sort_index()
        comb[sim_col] = comb[sim_col].interpolate(method="time")
        aligned = comb.loc[obs_clean.index, [sim_col]].join(obs_clean).dropna()
        aligned_data_dict[d] = aligned
        if len(aligned) > 0:
            rmse_val = np.sqrt(mean_squared_error(aligned[obs_col], aligned[sim_col]))
            mae_val = np.mean(np.abs(aligned[obs_col] - aligned[sim_col]))
            r2_val = r2_score(aligned[obs_col], aligned[sim_col])
            bias_val = np.mean(aligned[sim_col] - aligned[obs_col])
        else: rmse_val, mae_val, r2_val, bias_val = np.nan, np.nan, np.nan, np.nan
        performance_metrics.append({
            'Depth (m)': d, 'RMSE (°C)': rmse_val, 'MAE (°C)': mae_val,
            'Bias (°C)': bias_val, 'R²': r2_val, 'Data Points': len(aligned)
        })

In [ ]:
old_sim_dir = r'Calibration\3_MonthBest\dflowfm'
df_sim_old = read_and_interpolate_his(old_sim_dir, obs_station, depths_selected)
# Đảm bảo dữ liệu mô phỏng MỚI (Tối ưu) đã được đọc
if 'df_sim_final' not in locals() or df_sim_final.empty:
    df_sim_final = read_and_interpolate_his(final_dir, obs_station, depths_selected)
# 2. Xác định khoảng thời gian chung cho cả 3 chuỗi dữ liệu
t_min = max(temp_measured["TIMESTAMP"].min(), df_sim_final["TIMESTAMP"].min(), df_sim_old["TIMESTAMP"].min())
t_max = min(temp_measured["TIMESTAMP"].max(), df_sim_final["TIMESTAMP"].max(), df_sim_old["TIMESTAMP"].max())
obs_sub = temp_measured[(temp_measured["TIMESTAMP"] >= t_min) & (temp_measured["TIMESTAMP"] <= t_max)]
sim_new_sub = df_sim_final[(df_sim_final["TIMESTAMP"] >= t_min) & (df_sim_final["TIMESTAMP"] <= t_max)].set_index("TIMESTAMP")
sim_old_sub = df_sim_old[(df_sim_old["TIMESTAMP"] >= t_min) & (df_sim_old["TIMESTAMP"] <= t_max)].set_index("TIMESTAMP")
# 3. Tính toán các chỉ số thống kê chi tiết cho từng tầng độ sâu
comparison_metrics = []
aligned_all_dict = {}
for d in depths_selected:
    obs_col, sim_col = f"obs_{d}", f"sim_{d}"
    # Lấy và lọc ngoại lai số liệu thực đo
    obs_series = obs_sub[["TIMESTAMP", obs_col]].dropna().set_index("TIMESTAMP").sort_index()
    obs_series = obs_series[~obs_series.index.duplicated(keep='first')]
    obs_clean = remove_rolling_outliers(obs_series, obs_col)
    # Căn chỉnh chuỗi mô phỏng MỚI về mốc đo
    comb_new = sim_new_sub[[sim_col]].reindex(sim_new_sub.index.union(obs_clean.index)).sort_index()
    comb_new[sim_col] = comb_new[sim_col].interpolate(method="time")
    aligned_new = comb_new.loc[obs_clean.index, [sim_col]].rename(columns={sim_col: f"sim_new_{d}"})
    # Căn chỉnh chuỗi mô phỏng CŨ về mốc đo
    comb_old = sim_old_sub[[sim_col]].reindex(sim_old_sub.index.union(obs_clean.index)).sort_index()
    comb_old[sim_col] = comb_old[sim_col].interpolate(method="time")
    aligned_old = comb_old.loc[obs_clean.index, [sim_col]].rename(columns={sim_col: f"sim_old_{d}"})
    # Gộp chung vào DataFrame
    merged = obs_clean.join(aligned_old).join(aligned_new).dropna()
    aligned_all_dict[d] = merged
    if len(merged) > 0:
        rmse_old = np.sqrt(mean_squared_error(merged[obs_col], merged[f"sim_old_{d}"]))
        rmse_new = np.sqrt(mean_squared_error(merged[obs_col], merged[f"sim_new_{d}"]))
        mae_old = np.mean(np.abs(merged[obs_col] - merged[f"sim_old_{d}"]))
        mae_new = np.mean(np.abs(merged[obs_col] - merged[f"sim_new_{d}"]))
        r2_old = r2_score(merged[obs_col], merged[f"sim_old_{d}"])
        r2_new = r2_score(merged[obs_col], merged[f"sim_new_{d}"])
        improv = ((rmse_old - rmse_new) / rmse_old) * 100.0
    else: rmse_old, rmse_new, mae_old, mae_new, r2_old, r2_new, improv = [np.nan] * 7
    comparison_metrics.append({'Depth (m)': d, 'Old RMSE (°C)': rmse_old, 'New RMSE (°C)': rmse_new,
        'Old MAE (°C)': mae_old, 'New MAE (°C)': mae_new, 'Old R²': r2_old,
        'New R²': r2_new, 'Improvement (%)': improv
    })
df_comp = pd.DataFrame(comparison_metrics)
weighted_old_rmse = compute_weighted_rmse(depths_selected, df_comp['Old RMSE (°C)'].tolist(), rmse_method)
weighted_new_rmse = compute_weighted_rmse(depths_selected, df_comp['New RMSE (°C)'].tolist(), rmse_method)
total_improv = ((weighted_old_rmse - weighted_new_rmse) / weighted_old_rmse) * 100.0
print(df_comp.to_string(index=False))
print("---------------------------------------------------------------------")
print(f">> Total RMSE (Old - Baseline)   : {weighted_old_rmse:.4f} °C")
print(f">> Total RMSE (New - Optimized)  : {weighted_new_rmse:.4f} °C")
print(f">> Improvement                   : {total_improv:+.2f} %")
print("=====================================================================\n")
# 4. Vẽ biểu đồ chuỗi thời gian so sánh đa tầng bố cục 2 cột
sns.set_theme(style="ticks")
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 13,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 10, 'figure.titlesize': 15
})
n_depths = len(depths_selected)
n_cols = 2
n_rows = (n_depths + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4.2 * n_rows), sharex=True)
axes = axes.flatten() if n_depths > 1 else [axes]
for idx, d in enumerate(depths_selected):
    ax = axes[idx]
    obs_col, sim_col = f"obs_{d}", f"sim_{d}"
    # 1. Đường mô phỏng CŨ (Baseline)
    if not df_sim_old.empty:
        ax.plot(df_sim_old['TIMESTAMP'], df_sim_old[sim_col], 
            label='Old Simulation (Baseline)', color='#7f7f7f', linestyle='--', linewidth=1.5, alpha=0.85)
    # 2. Đường mô phỏng MỚI (Sau hiệu chỉnh Optuna)
    if not df_sim_final.empty:
        ax.plot(df_sim_final['TIMESTAMP'], df_sim_final[sim_col], 
            label='New Simulation (Optimized)', color='#d62728', linestyle='-', linewidth=1.8, alpha=0.9)
    # 3. Điểm số liệu THỰC ĐO (Profiler)
    if d in aligned_all_dict and not aligned_all_dict[d].empty:
        merged = aligned_all_dict[d]
        ax.plot(merged.index, merged[obs_col], 
            label='Observation (Profiler)', color='#1f77b4', linestyle='-', 
            linewidth=1.4, marker='o', markersize=2.5, alpha=0.85, zorder=3)
    # Box thông tin sai số
    row_info = df_comp[df_comp['Depth (m)'] == d].iloc[0]
    box_text = (f"Old RMSE: {row_info['Old RMSE (°C)']:.2f}°C  (R²: {row_info['Old R²']:.2f})\n"
                f"New RMSE: {row_info['New RMSE (°C)']:.2f}°C (R²: {row_info['New R²']:.2f})\n"
                f"Improvement: {row_info['Improvement (%)']:+.1f}%")
    ax.text(0.02, 0.95, box_text, transform=ax.transAxes, va='top',
            fontsize=9.5, bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85, edgecolor="#cccccc"))
    ax.set_title(f"Water Depth: {d} m", fontweight='bold')
    ax.set_ylabel("Temperature (°C)")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc='upper right')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m'))
# Ẩn các ô trống nếu số độ sâu là số lẻ
for idx in range(n_depths, len(axes)):
    fig.delaxes(axes[idx])
plt.suptitle(f"Temperature Comparation (Station: {obs_station}) - Method for RMSE computation: {rmse_method}\n"
             f"Total RMSE: Old = {weighted_old_rmse:.3f}°C -> New = {weighted_new_rmse:.3f}°C (Improvement {total_improv:+.1f}%)", 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()
# 5. Đồ thị tương quan Parity Plot 1:1 so sánh 2 trường hợp
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), sharex=True, sharey=True)
# Lấy giá trị Min-Max
all_obs = np.concatenate([aligned_all_dict[d][f"obs_{d}"].values for d in depths_selected if d in aligned_all_dict])
all_sim_old = np.concatenate([aligned_all_dict[d][f"sim_old_{d}"].values for d in depths_selected if d in aligned_all_dict])
all_sim_new = np.concatenate([aligned_all_dict[d][f"sim_new_{d}"].values for d in depths_selected if d in aligned_all_dict])
min_v = min(all_obs.min(), all_sim_old.min(), all_sim_new.min()) - 0.5
max_v = max(all_obs.max(), all_sim_old.max(), all_sim_new.max()) + 0.5
colors = sns.color_palette("tab10", n_depths)
# Ô bên trái: Mô phỏng Cũ vs Thực đo
for idx, d in enumerate(depths_selected):
    if d in aligned_all_dict:
        m = aligned_all_dict[d]
        axes[0].scatter(m[f"obs_{d}"], m[f"sim_old_{d}"], label=f"Depth {d} m", color=colors[idx], alpha=0.5, s=20)
axes[0].plot([min_v, max_v], [min_v, max_v], 'k--', lw=1.5, label='Line 1:1')
axes[0].set_title(f"Before Calibration (Baseline)\nTotal RMSE = {weighted_old_rmse:.3f} °C", fontweight='bold')
axes[0].set_xlabel("Observation (°C)")
axes[0].set_ylabel("Simulation (°C)")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend(loc='lower right', fontsize=9)
# Ô bên phải: Mô phỏng Mới vs Thực đo
for idx, d in enumerate(depths_selected):
    if d in aligned_all_dict:
        m = aligned_all_dict[d]
        axes[1].scatter(m[f"obs_{d}"], m[f"sim_new_{d}"], label=f"Depth {d} m", color=colors[idx], alpha=0.5, s=20)
axes[1].plot([min_v, max_v], [min_v, max_v], 'k--', lw=1.5, label='Line 1:1')
axes[1].set_title(f"After Calibration (Optimized)\nTotal RMSE = {weighted_new_rmse:.3f} °C", fontweight='bold')
axes[1].set_xlabel("Observation (°C)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend(loc='lower right', fontsize=9)
axes[0].set_xlim(min_v, max_v)
axes[0].set_ylim(min_v, max_v)
plt.tight_layout()
plt.show()

In [ ]:
import os, shutil, subprocess, sys, pickle, optuna, logging, warnings
sys.path.append('backend/app/')
import pandas as pd, xarray as xr, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import qmc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, WhiteKernel
from sklearn.gaussian_process.kernels import ConstantKernel as C
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.model_selection import KFold
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, r2_score
from functools import partial

In [ ]:
# Setup functions
def check_remove_mdu_lines(mdu_data, variable, value=''):
    mdu = mdu_data.copy()
    index, line = next((i, line) for i, line in enumerate(mdu) if line.strip().startswith(variable))
    new = line.split('=')
    new1 = new[1].split('#')
    mdu[index] = f'{new[0]}= {value.ljust(len(new1[0])-2)} #{new1[1]}'
    return mdu

def rmse_weight(depths:list, rmse_values:list, method:str):
    depth_arr, rmse_array = np.array(depths), np.array(rmse_values)
    valid_mask = ~np.isnan(rmse_array)
    if not valid_mask.any(): return np.nan
    # Get valid depths and RMSE
    valid_depths = depth_arr[valid_mask]
    valid_rmse = rmse_array[valid_mask]
    # Calculate weights based on method
    if method == 'surface': weights = 1.0 / valid_depths
    elif method == 'uniform': weights = np.ones(len(valid_depths))
    elif method == 'balanced': weights = np.log(valid_depths + 1)
    elif method == 'deep': weights = valid_depths
    # Normalize weights to sum to 1
    weights = weights / np.sum(weights)
    # Calculate weighted RMSE
    rmse = np.sqrt(np.sum(weights * valid_rmse**2))
    return rmse

def run(mdu_path, path, dir):
    bat_path = os.path.normpath(os.path.join(dir, r"backend\softs\x64\dflowfm\scripts\run_dflowfm.bat"))
    new_path = os.path.normpath(os.path.join(dir, mdu_path))
    command = ["cmd.exe", "/c", bat_path, "--autostartstop", new_path]
    print("=" * 80)
    print(f"[STARTING] {mdu_path}")
    print("=" * 80)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1, cwd=path)
    error_detected = False
    # Stream logs
    for line in process.stdout:
        line = line.strip()
        if not line: continue
        print(line)
        # Catch error messages
        if "forrtl:" in line.lower() or "error" in line.lower(): error_detected = True
    return_code = process.wait()
    print("=" * 80)
    if return_code == 0 and not error_detected:
        print(f"[COMPLETED] {mdu_path}")
        print("=" * 80)
        return True
    else:
        print(f"[FAILED] {mdu_path}, exit code={return_code}")
        print("=" * 80)
        return False

def interpolate_profile_time(group, target_depths):
    group = group.sort_values("depth").drop_duplicates("depth").reset_index(drop=True)
    # t_start, t_end = group["TIMESTAMP"].iloc[0], group["TIMESTAMP"].iloc[-1]
    depths, temperatures = group["depth"].to_numpy(), group["temperature"].to_numpy()
    result, time = [], group["TIMESTAMP"].iloc[0]
    time_seconds = (group["TIMESTAMP"] - time).dt.total_seconds().to_numpy()
    for target_depth in target_depths:
        if not (depths.min() <= target_depth <= depths.max()): continue
        # Interpolate temperature at the target depth
        temperature = np.interp(target_depth, depths, temperatures)
        elapsed_seconds = np.interp(target_depth, depths, time_seconds)
        timestamp = time + pd.to_timedelta(elapsed_seconds, unit="s").round("1s")
        row = { "TIMESTAMP": timestamp,
            # "profile_start": t_start, "profile_end": t_end
        }
        for depth in target_depths:
            row[f"obs_{depth}"] = np.nan
        row[f"obs_{target_depth}"] = temperature
        result.append(row)
    return pd.DataFrame(result)

def interpolate_temperature(temp_row, depth, target_depths):
    mask = np.isfinite(temp_row) & np.isfinite(depth)
    if mask.sum() < 2:
        return np.full(len(target_depths), np.nan)
    d, t = depth[mask], temp_row[mask]
    order = np.argsort(d)
    d, t = d[order], t[order]
    # Interpolate temperature at the target depths
    result = np.full(len(target_depths), np.nan)
    valid = (target_depths >= d.min()) & (target_depths <= d.max())
    result[valid] = np.interp(target_depths[valid], d, t)
    return result

# Remove outliers from the measured data using rolling window method
def remove_rolling_outliers(df, column, window=20, n_std=3, center=True, min_periods=None):
    df = df.copy()
    if min_periods is None: min_periods = max(3, window // 2)
    rolling_mean = df[column].rolling(window=window, center=center, min_periods=min_periods).mean()
    rolling_std = df[column].rolling(window=window, center=center, min_periods=min_periods).std()
    lower_bound = rolling_mean - n_std * rolling_std
    upper_bound = rolling_mean + n_std * rolling_std
    mask = (df[column] >= lower_bound) & (df[column] <= upper_bound)
    mask = mask | df[column].isna()
    df_filtered = df[mask].copy()
    return df_filtered

def output_reader(folder, obs_name, depths_selected, layer_path=None):
    model_dir = os.path.join(folder, 'output')
    his_path = os.path.join(model_dir, 'FlowFM_his.nc')
    map_path = os.path.join(model_dir, 'FlowFM_map.nc')
    if not os.path.exists(his_path): print('Cannot find his.nc file')
    his_ds = xr.open_dataset(his_path)
    station_names = [name.decode('utf-8').strip() for name in his_ds['station_name'].values]
    # Get id of the station with the name 'Profiler'
    name_id = station_names.index(obs_name) if obs_name in station_names else None
    if name_id == None: print(f'Cannot find station: {obs_name}')
    # Get the simulated temperature data for the station
    temp_df = his_ds['temperature'].values[:, name_id, :]
    if os.path.exists(map_path):
        water_level = his_ds['waterlevel'].values[name_id][0]
        map_ds = xr.open_dataset(map_path)
        depth_layers = map_ds['mesh2d_layer_z'].values
        depth_adjusted = water_level - depth_layers
        with open(layer_path, "w", encoding="utf-8") as f:
            json.dump(list(depth_adjusted), f, ensure_ascii=False, indent=4)
        map_ds.close()
    his_ds.close()
    if os.path.exists(layer_path):
        with open(layer_path, 'r', encoding="utf-8") as f:
            depth_adjusted = np.array(json.load(f))
    temp_interp = np.array([
        interpolate_temperature(temp_df[i], depth_adjusted, np.array(depths_selected))
        for i in range(temp_df.shape[0])
    ])
    temp_sim = pd.DataFrame(
        temp_interp, columns=[f"sim_{depth}" for depth in depths_selected],
        index=his_ds['time'].values).reset_index().rename(columns={"index": "TIMESTAMP"}
    )
    return temp_sim

def scenario_creator(folder, df_params, mdu_path, mdu, his_interval, single=False):
    custom_params = ['CloudFactor', 'CloudOffset', 'AirTemperature']
    for id, row in df_params.iterrows():
        row_dict, mdu_updated = row.to_dict(), mdu.copy()
        for idx, value in row_dict.items():
            if idx in custom_params: continue
            if isinstance(value, float): value_str = f"{value:.5e}"
            else: value_str = str(value)
            mdu_updated = check_remove_mdu_lines(mdu_updated, idx, value_str)
        # Create _map.nc file for the first simulation
        if not single:
            if id != 1: mdu_updated = check_remove_mdu_lines(mdu_updated, 'MapInterval', '0')
            new_dir = os.path.join(folder, str(id))
        else:
            mdu_updated = check_remove_mdu_lines(mdu_updated, 'MapInterval', his_interval)
            new_dir = folder
        # Write file
        if not os.path.exists(new_dir): os.makedirs(new_dir)
        parent_dir = os.path.dirname(mdu_path)
        files = [f for f in os.listdir(parent_dir) if os.path.isfile(os.path.join(parent_dir, f))]
        for file in files:
            if file.endswith('.mdu'): continue
            src_path = os.path.join(parent_dir, file)
            dst_path = os.path.join(new_dir, file)
            shutil.copy(src_path, dst_path)
        # Adjust cloudiness and air temperature values
        meteo_path = os.path.join(new_dir, 'FlowFM_meteo.tim')
        if os.path.exists(meteo_path):
            meteo_data = np.loadtxt(meteo_path)
            # Cloudiness
            factor = row_dict.get('CloudFactor', 1.0)
            offset = row_dict.get('CloudOffset', 0.0)
            original_cloud = meteo_data[:, 3]
            new_cloud = np.clip(original_cloud * factor + offset, 0, 100)
            meteo_data[:, 3] = new_cloud
            # Air temperature
            air = row_dict.get('AirTemperature', 0.0)
            original_air = meteo_data[:, 2]
            new_air = original_air - air
            meteo_data[:, 2] = new_air
            np.savetxt(meteo_path, meteo_data, fmt='%.7e')
        # Write the new mdu file
        path = os.path.join(new_dir, 'FlowFM.mdu')
        with open(path, 'w') as file:
            file.write(''.join(mdu_updated))

# Create data for trainning model

### Initial parameters

In [ ]:
# Init parameters
n_run, test_folder = 60, r'Calibration\test'
param_path = f'{test_folder}/parameter_samples.csv'
his_interval = 3600*6  # 1/2 day in seconds
mdu_path = r'Calibration\3_MonthBest\dflowfm\FlowFM.mdu'
dir = r'C:\Users\vanln\Downloads\Hydro-AI-Platform'
obs_name, depths_selected = 'Profiler', [5, 10, 30, 40, 50]
parameters = {
    'Secchidepth': [0.3, 10], 'Stanton': [0.001, 0.0016], 'Dalton': [0.001, 0.0016],
    'Vicoww': [0, 1e-4], 'Dicoww': [0, 1e-4], 'Vicouv': [0.1, 2.0], 'Dicouv': [0.1, 2.0],
    'CloudFactor': [0.5, 1.5], 'CloudOffset': [-20, 20], # Cloudness
    'AirTemperature': [-5, 5]
    # 'InitialTemperature': [2.0, 6.0], 'UnifFrictCoef': [0.015, 0.035] # Manning n
}

### Measured data

In [ ]:
# Read observations
measured_path = r'Calibration\Profiler_modem_PFL_Step.dat'
measured_df = pd.read_csv(measured_path, delimiter=',', header=0, skiprows=[0, 2, 3], low_memory=False)
measured_df["TIMESTAMP"] = pd.to_datetime(measured_df["TIMESTAMP"])
measured_df["temperature"] = pd.to_numeric(measured_df["sensorParms(1)"], errors='coerce')
measured_df["depth"] = pd.to_numeric(measured_df["sensorParms(9)"], errors='coerce')
measured_df = measured_df[["TIMESTAMP", "temperature", "depth"]]
# interpolate missing values in the measured data
df_interpolated = measured_df.interpolate(method='linear', limit_direction='both')
# Get data for the year 2024
measured_2024 = df_interpolated[df_interpolated["TIMESTAMP"].dt.year == 2024].reset_index(drop=True)
df = (measured_2024.sort_values("TIMESTAMP").reset_index(drop=True))
new_profile = ((df["depth"] < 1.5) & (df["depth"].shift(1) > 5))
df['profile'] = new_profile.cumsum()
temp_measured = (df.groupby("profile").apply(lambda g: interpolate_profile_time(g, depths_selected))
                 .reset_index(drop=True).sort_values("TIMESTAMP").reset_index(drop=True))

### Prepare sample datasets

In [ ]:
# Create Latin Hypercube Sampling (LHS) design for the parameters
param_names = list(parameters.keys())
l_bounds = [parameters[p][0] for p in param_names]
u_bounds = [parameters[p][1] for p in param_names]
sampler = qmc.LatinHypercube(d=len(param_names))
samples = sampler.random(n=n_run)
samples_scaled = qmc.scale(samples, l_bounds, u_bounds)
df_params = pd.DataFrame(samples_scaled, columns=param_names, index=np.arange(1, samples_scaled.shape[0]+1))
df_params.to_csv(param_path)

In [ ]:
# Read mdu file
with open(mdu_path, 'r') as f:
    mdu = f.readlines()
mdu = check_remove_mdu_lines(mdu, 'WaqInterval', '0')
mdu = check_remove_mdu_lines(mdu, 'MapInterval', str(his_interval))
mdu = check_remove_mdu_lines(mdu, 'RstInterval', '0')
mdu = check_remove_mdu_lines(mdu, 'RestartFile', '')
mdu = check_remove_mdu_lines(mdu, 'RestartDateTime', '')
# Change the his_interval in the mdu file
mdu = check_remove_mdu_lines(mdu, 'HisInterval', str(his_interval))
# Change value of AntiCreep to 1
mdu = check_remove_mdu_lines(mdu, 'AntiCreep', '1')
mdu = check_remove_mdu_lines(mdu, 'Kmx', '10')
new_line = 'verticalAdvectionType             = 4               #\n'
new_line_normalized = new_line.strip().lower()
if not any(line.strip().lower() == new_line_normalized for line in mdu):
    for i, line in enumerate(mdu):
        if line.strip().lower() == '[physics]':
            mdu.insert(i + 1, new_line)
            break
# Optimize writing variables in his file
variables_to_keep = {'Wrihis_temperature', 'Wrihis_waterlevel_s1'}
variables_to_update = {
    line.split('=')[0].strip()
    for line in mdu if line.strip().startswith(('Wrihis_', 'Wrimap_'))
    and line.split('=')[0].strip() not in variables_to_keep
}
for var in variables_to_update:
    mdu = check_remove_mdu_lines(mdu, var, '0')
for var in variables_to_keep:
    mdu = check_remove_mdu_lines(mdu, var, '1')

In [ ]:
# Create different scenarios
scenario_dir = os.path.join(test_folder, 'scenarios')
if not os.path.exists(scenario_dir): os.makedirs(scenario_dir)
scenario_creator(scenario_dir, df_params, mdu_path, mdu, str(his_interval))

In [ ]:
# meteo_path = r'Calibration\test\scenarios\500\FlowFM_meteo.tim'
# meteo_data = np.loadtxt(r'Calibration\3_MonthBest\dflowfm\FlowFM_meteo.tim')
# original_air = meteo_data[:, 2]
# new_air = original_air + 4
# meteo_data[:, 2] = new_air
# np.savetxt(meteo_path, meteo_data, fmt='%.7e')
# for item in ['500']:
#     print(f'\nRunning model: {item}...')
#     model_dir = os.path.join(test_folder, 'scenarios', item)
#     model_path = os.path.join(model_dir, 'FlowFM.mdu')
#     success = run(model_path, model_dir, dir)
#     print(success)
# folders = ['500']

In [ ]:
# Get all scenario's names
scenario_dir = os.path.join(test_folder, 'scenarios')
folders = sorted([x for x in os.listdir(scenario_dir) if os.path.isdir(os.path.join(scenario_dir, x))], key=int)

### Run cases

In [ ]:
# Run all scenarios using Delft3D FM
err_path = os.path.join(test_folder, 'models_error.txt')
if os.path.exists(err_path): os.remove(err_path)
err_models = []
for item in folders:
    print(f'\nRunning model: {item}...')
    model_dir = os.path.join(test_folder, 'scenarios', item)
    model_path = os.path.join(model_dir, 'FlowFM.mdu')
    success = run(model_path, model_dir, dir)
    if not success:
        print(f"Model '{item}' failed.")
        err_models.append(item)
print("All simulations finished.")
print(f'Number of error models: {len(err_models)}')
if len(err_models) > 0:
    with open(err_path, 'w') as file:
        file.write('\n'.join(err_models))

### Prepare outputs

In [ ]:
# Read outputs and format data
result, layer_path = pd.DataFrame(), f"{test_folder}/depth_layers.json"
for item in folders:
    model_dir = os.path.join(test_folder, 'scenarios', item)
    temp = output_reader(model_dir, obs_name, depths_selected, layer_path)
    temp['case'] = item
    result = pd.concat([result, temp], axis=0)

In [ ]:
# Select period
obs_start, obs_end = temp_measured["TIMESTAMP"].min(), temp_measured["TIMESTAMP"].max()
sim_start, sim_end = result["TIMESTAMP"].min(), result["TIMESTAMP"].max()
period_start, period_end = max(obs_start, sim_start), min(obs_end, sim_end)
obs_clip = temp_measured[
    (period_start <= temp_measured['TIMESTAMP']) & (temp_measured['TIMESTAMP'] <= period_end)
]
sim_clip = result[(period_start <= result['TIMESTAMP']) & (result['TIMESTAMP'] <= period_end)]

In [ ]:
summary, weight = pd.DataFrame(), True
method = 'deep' # 'balanced', 'surface', , 'uniform'
params = pd.read_csv(param_path, index_col=0)
for i in folders:
    clip = sim_clip[sim_clip['case']==str(i)]
    rmse, bias = [], []
    for layer_depth in depths_selected:
        obs_col, sim_col = f'obs_{layer_depth}', f'sim_{layer_depth}'
        obs = obs_clip[['TIMESTAMP', obs_col]].dropna().reset_index(drop=True)
        sim = clip[["TIMESTAMP", sim_col]].dropna().reset_index(drop=True)
        sim["TIMESTAMP"] = pd.to_datetime(sim["TIMESTAMP"])
        obs["TIMESTAMP"] = pd.to_datetime(obs["TIMESTAMP"])
        sim = sim.set_index("TIMESTAMP").sort_index()
        sim = sim[~sim.index.duplicated(keep='first')]
        obs = obs.set_index("TIMESTAMP").sort_index()
        obs = obs[~obs.index.duplicated(keep='first')]
        # Remove outliers from the observed temperature data
        obs_filled = remove_rolling_outliers(obs, obs_col)
        combined = sim.reindex(sim.index.union(obs_filled.index)).sort_index()
        # Interpolate the simulated temperature to match the observation timestamps
        combined[sim_col] = combined[sim_col].interpolate(method="time")
        result_combined = combined.loc[obs_filled.index, [sim_col]]
        obs_filled.rename(columns={sim_col: obs_col}, inplace=True)
        result_combined = result_combined.join(obs_filled)
        result_combined = result_combined.reset_index().set_index("TIMESTAMP")
        if len(result_combined) > 0:
            rmse_temp = ((result_combined[obs_col] - result_combined[sim_col]) ** 2).mean() ** 0.5
            rmse.append(rmse_temp)
            bias_temp = np.mean(result_combined[obs_col] - result_combined[sim_col])
            bias.append(bias_temp)
        else: rmse.append(np.nan)
    # Check if one of values of rmse is nan
    if not pd.isna(rmse).any():
        if weight:
            rmse_mean = rmse_weight(depths_selected, rmse, method)
        else: rmse_mean = np.array(rmse).mean()
    else: rmse_mean = np.nan
    df_temp = params.iloc[int(i)-1:int(i)].copy()
    df_temp['RMSE'] = rmse_mean
    # print(f'{i}: bias - {bias}')
    # print(f'{i}: rmse - {rmse}')
    # print(f'{i}: rmse_mean - {rmse_mean}')
    summary = pd.concat([summary, df_temp])

In [ ]:
summary.head()

In [ ]:
summary.describe()

# Process

In [ ]:
# Correlation
correlation = summary.corr()['RMSE'].sort_values(ascending=False)
print(correlation)

plt.figure(figsize=(12, 8))
sns.heatmap(summary.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(summary['RMSE'], bins=10, edgecolor='black', alpha=0.7)
plt.xlabel('RMSE (°C)')
plt.ylabel('Number of iterration')
plt.title('MSE Distribution')
plt.grid(True, alpha=0.3)
plt.subplot(1, 2, 2)
plt.scatter(range(len(summary)), summary['RMSE'], alpha=0.7)
plt.xlabel('Run')
plt.ylabel('RMSE (°C)')
plt.title('RMSE')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
X, y = summary.drop(columns=['RMSE']), summary[['RMSE']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_X, scaler_y = StandardScaler(), StandardScaler()
X_scaled_fitted, y_scaled_fitted = scaler_X.fit(X_train), scaler_y.fit(y_train)
X_train_scaled = X_scaled_fitted.transform(X_train)
y_train_scaled = y_scaled_fitted.transform(y_train).ravel()
X_test_scaled = X_scaled_fitted.transform(X_test)
# Save scalers
folder_temp = f'{test_folder}/model'
if not os.path.exists(folder_temp): os.makedirs(folder_temp)
for scaler, data in [('scaler_X', X_scaled_fitted), ('scaler_y', y_scaled_fitted)]:
    with open(f'{folder_temp}/{scaler}.pkl', "wb") as f:
        pickle.dump(data, f)

### Gausian

In [ ]:
def gpr_kernel_cv(X_train, y_train, kernel, kernel_name, n_folds=5):
    try:
        gpr = GaussianProcessRegressor(
            kernel=kernel, n_restarts_optimizer=30,
            alpha=1e-6, normalize_y=True, random_state=42
        )
        kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
        rmse_scores, r2_scores = [], []
        for train_idx, val_idx in kf.split(X_train):
            X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
            y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
            # Clone models
            gpr_fold = clone(gpr)
            gpr_fold.fit(X_train_fold, y_train_fold)
            y_pred = gpr_fold.predict(X_val_fold)
            rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred))
            rmse_scores.append(rmse)
            r2 = r2_score(y_val_fold, y_pred)
            r2_scores.append(r2)
        mean_rmse, std_rmse, r2_mean = np.mean(rmse_scores), np.std(rmse_scores), np.mean(r2_scores)
        return {'kernel': kernel_name, 'kernel_obj': kernel, 'r2_mean': r2_mean,
                'rmse_mean': mean_rmse, 'rmse_std': std_rmse
            }
    except Exception as e:
        print(f"Error kernel {kernel_name}: {e}")
        return None
kernels_to_test = [
    (C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-3, 1e2)), 'RBF'),
    (C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2)) + WhiteKernel(1e-3, (1e-10, 1e-1)), 'RBF + White'),
    (C(1.0, (1e-3, 1e3)) * Matern(1.0, nu=1.5, length_scale_bounds=(1e-3, 1e2)), 'Matern_1.5'),
    (C(1.0, (1e-3, 1e3)) * Matern(1.0, nu=2.5, length_scale_bounds=(1e-3, 1e2)), 'Matern_2.5'),
    (C(1.0, (1e-3, 1e3)) * RationalQuadratic(1.0, 1.0, length_scale_bounds=(1e-9, 1e6)), 'RationalQuadratic'),
]
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn.gaussian_process.kernels')
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')
results_cv = []
for kernel, name in kernels_to_test:
    result = gpr_kernel_cv(X_train_scaled, y_train_scaled, kernel, name, 5)
    if result: results_cv.append(result)
best_result_cv = min(results_cv, key=lambda x: x['rmse_mean'])
print(f"\nThe best Kernel: {best_result_cv['kernel']}")
print(f"   RMSE: {best_result_cv['rmse_mean']:.4f} ± {best_result_cv['rmse_std']:.4f}  - R2: {best_result_cv['r2_mean']:.4f}")

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
rf_default = RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1
)
rf_scores = cross_val_score(
    rf_default, X_train_scaled, y_train_scaled, cv=5, scoring='r2'
)
rf_rmse = np.sqrt(-cross_val_score(
    rf_default, X_train_scaled, y_train_scaled, 
    cv=5, scoring='neg_mean_squared_error'
))
print(f"\nRandom Forest (default):")
print(f"  R²: {rf_scores.mean():.4f} ± {rf_scores.std():.4f}")
print(f"  RMSE: {rf_rmse.mean():.4f} ± {rf_rmse.std():.4f}")

In [ ]:
results_cv

In [ ]:
# Testing
final_gpr = GaussianProcessRegressor(
    kernel=best_result_cv['kernel_obj'], n_restarts_optimizer=10,
    alpha=1e-6, normalize_y=True, random_state=42
)
final_gpr.fit(X_train_scaled, y_train_scaled)
y_pred_test = final_gpr.predict(X_test_scaled)
y_pred_original = scaler_y.inverse_transform(y_pred_test.reshape(-1, 1)).ravel()
rmse = np.sqrt(mean_squared_error(y_pred_original, y_test))
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_original, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], 
         [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Real value (RMSE)')
plt.ylabel('Predicted value (RMSE)')
plt.title(f'GPR: Prediction vs. Reality\nRMSE = {rmse:.4f}°C')
plt.subplot(1, 2, 2)
residuals = np.array(y_test).flatten() - y_pred_original
plt.scatter(y_pred_original, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Value')
plt.ylabel('Residuals')
plt.title('Residual Chart')
plt.tight_layout()
plt.show()

In [ ]:
# Use the best Kernel to train on all dataset
final_kernel = best_result_cv['kernel_obj']
gpr_final = GaussianProcessRegressor(
    kernel=best_result_cv['kernel_obj'], n_restarts_optimizer=30,
    alpha=1e-6, normalize_y=True, random_state=42
)
X_scaled = X_scaled_fitted.transform(X)
y_scaled = y_scaled_fitted.transform(y).ravel()
gpr_final.fit(X_scaled, y_scaled)
# Save model
model_path = f'{folder_temp}/gpr_model.pkl'
with open(model_path, "wb") as f:
    pickle.dump(gpr_final, f)
print(f"Saved model to: '{model_path}'")

In [ ]:
# Run optimization with Optuna
def objective(trial, scaler_X, scaler_y, model):
    secchidepth = trial.suggest_float('Secchidepth', 2, 20)
    stanton = trial.suggest_float('Stanton', 0.0005, 0.002)
    dalton = trial.suggest_float('Dalton', 0.0005, 0.002)
    vicoww = trial.suggest_float('Vicoww', 1e-6, 1e-4, log=True)
    dicoww = trial.suggest_float('Dicoww', 1e-6, 1e-4, log=True)
    vicouv = trial.suggest_float('Vicouv', 0.1, 5.0)
    cloud_factor = trial.suggest_float('CloudFactor', 0.5, 1.5)
    cloud_offset = trial.suggest_float('CloudOffset', -20, 20)
    air_temp = trial.suggest_float('AirTemperature', -5, 5)
    # Create input vector
    X = pd.DataFrame([[
        secchidepth, stanton, dalton, vicoww, dicoww, vicouv,
        cloud_factor, cloud_offset, air_temp
    ]], columns=[
        'Secchidepth', 'Stanton', 'Dalton', 'Vicoww', 'Dicoww', 
        'Vicouv', 'CloudFactor', 'CloudOffset', 'AirTemperature'
    ])
    X_scaled = scaler_X.transform(X)
    y_pred = model.predict(X_scaled)
    y_pred_original = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).ravel()
    return y_pred_original[0]

# Load scalers and model
objs = {}
for name in ['scaler_X', 'scaler_y', 'gpr_model']:
     with open(os.path.join(folder_temp, f"{name}.pkl"), 'rb') as f:
        objs[name] = pickle.load(f)

objective_with_params = partial(
    objective, scaler_X=objs['scaler_X'], scaler_y=objs['scaler_y'], model=objs['gpr_model']
)

study = optuna.create_study(
    sampler=optuna.samplers.TPESampler(seed=42),
    direction='minimize', study_name='delft3d_calibration'
)
logging.getLogger('optuna').setLevel(logging.WARNING)
print("Running Optuna...")
study.optimize(objective_with_params, n_trials=2000, show_progress_bar=True)

In [ ]:
print(f"Smallest RMSE: {study.best_value:.4f}°C")
print(f"\nOptimal parameters:")
for key, value in study.best_params.items():
    if isinstance(value, float):
        if value < 0.001: print(f"  {key}: {value:.6e}")
        else: print(f"  {key}: {value:.6f}")
    else: print(f"  {key}: {value}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax = axes[0, 0]
ax.plot(study.trials_dataframe()['value'].cummin(), 'b-', lw=2)
ax.set_xlabel('Iterations')
ax.set_ylabel('Smallest RMSE (°C)')
ax.set_title("Optina's convergence process")
ax.grid(True)
ax = axes[0, 1]
ax.hist(study.trials_dataframe()['value'], bins=20, alpha=0.7, color='blue')
ax.axvline(study.best_value, color='red', linestyle='--', lw=2, label=f'Best: {study.best_value:.4f}')
ax.set_xlabel('RMSE (°C)')
ax.set_ylabel('Iteration')
ax.set_title('RMSE Distribution')
ax.legend()
ax.grid(True)
ax = axes[1, 0]
importance = optuna.importance.get_param_importances(study)
param_names = list(importance.keys())
param_importance = list(importance.values())
colors = plt.cm.viridis(np.linspace(0, 1, len(param_names)))
ax.barh(param_names, param_importance, color=colors)
ax.set_xlabel('Importance')
ax.set_title('Importance of Parameters')
ax = axes[1, 1]
df_trials = study.trials_dataframe()
var = 'Secchidepth'
if f'params_{var}' in df_trials.columns:
    ax.scatter(df_trials[f'params_{var}'], df_trials['value'], alpha=0.5, s=10)
    ax.axvline(study.best_params[var], color='red', linestyle='--', 
               label=f"Best: {study.best_params[var]:.2f}")
    ax.set_xlabel(var)
    ax.set_ylabel('RMSE (°C)')
    ax.set_title(f'Sensitivity with {var}')
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Rerun Delft3D FM with the optimal parameters
folder_temp, objs = os.path.join(test_folder, 'model'), {}
best_folder = os.path.join(folder_temp, 'final')
if os.path.exists(best_folder): shutil.rmtree(best_folder)
os.mkdir(best_folder)
# Create a new scenario with the optimal parameters
scenario_creator(best_folder, pd.DataFrame([study.best_params]), mdu_path, mdu, True)
if not os.path.exists(best_folder): os.makedirs(best_folder)
for name in ['scaler_X', 'scaler_y', 'gpr_model']:
     with open(os.path.join(folder_temp, f"{name}.pkl"), 'rb') as f:
        objs[name] = pickle.load(f)
scaler_X, scaler_y, model = objs['scaler_X'], objs['scaler_y'], objs['gpr_model']
best_path = os.path.join(best_folder, 'FlowFM.mdu')
# Run a simulation
success = run(best_path, best_folder, dir)
print(success)

In [ ]:
# Read output
layer_path = f"{test_folder}/depth_layers.json"
old_folder = r'Calibration\3_MonthBest\dflowfm'
old_sim = output_reader(old_folder, obs_name, depths_selected, layer_path)
best_folder = os.path.join(folder_temp, 'final')
new_sim = output_reader(best_folder, obs_name, depths_selected, layer_path)
# Select period
obs_start, obs_end = temp_measured["TIMESTAMP"].min(), temp_measured["TIMESTAMP"].max()
old_start, old_end = old_sim["TIMESTAMP"].min(), old_sim["TIMESTAMP"].max()
new_start, new_end = new_sim["TIMESTAMP"].min(), new_sim["TIMESTAMP"].max()
period_start, period_end = max(obs_start, new_start, old_start), min(obs_end, new_end, old_end)
obs_clip = temp_measured[
    (period_start <= temp_measured['TIMESTAMP']) & (temp_measured['TIMESTAMP'] <= period_end)
]
old_sim_clip = old_sim[(period_start <= old_sim['TIMESTAMP']) & (old_sim['TIMESTAMP'] <= period_end)]
new_sim_clip = new_sim[(period_start <= new_sim['TIMESTAMP']) & (new_sim['TIMESTAMP'] <= period_end)]

# Compare new output with original
n_depths, n_cols = len(depths_selected), 2
n_rows = (n_depths + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
if n_rows == 1: axes_flat = [axes]
else: axes_flat = axes.flatten()
for i, depth in enumerate(depths_selected):
    ax = axes_flat[i]
    obs_col, old_col, new_col = f'obs_{depth}', f'sim_{depth}', f'sim_{depth}'
    obs = obs_clip[['TIMESTAMP', obs_col]].dropna().reset_index(drop=True)
    sim_old = old_sim_clip[["TIMESTAMP", old_col]].dropna().reset_index(drop=True)
    sim_new = new_sim_clip[["TIMESTAMP", new_col]].dropna().reset_index(drop=True)
    obs = obs.set_index("TIMESTAMP").sort_index()
    obs = obs[~obs.index.duplicated(keep='first')]
    sim_old = sim_old.set_index("TIMESTAMP").sort_index()
    sim_old = sim_old[~sim_old.index.duplicated(keep='first')]
    sim_new = sim_new.set_index("TIMESTAMP").sort_index()
    sim_new = sim_new[~sim_new.index.duplicated(keep='first')]
    # # Remove outliers from the observed temperature data
    # obs = remove_rolling_outliers(obs, obs_col)
    combined_old = sim_old.reindex(sim_old.index.union(obs.index)).sort_index()
    combined_new = sim_new.reindex(sim_new.index.union(obs.index)).sort_index()
    # Interpolate the simulated temperature to match the observation timestamps
    combined_old[old_col] = combined_old[old_col].interpolate(method="time")
    combined_new[new_col] = combined_new[new_col].interpolate(method="time")
    combined_old.rename(columns={old_col: 'sim_old'}, inplace=True)
    combined_new.rename(columns={new_col: 'sim_new'}, inplace=True)
    combine = obs.join([combined_old, combined_new])
    combine = combine.dropna()
    rmse_old = np.sqrt(mean_squared_error(combine[obs_col], combine['sim_old']))
    rmse_new = np.sqrt(mean_squared_error(combine[obs_col], combine['sim_new']))
    ax.plot(combine.index, combine[obs_col], 'o-', color='green', markersize=4, label='Measurement', linewidth=2.0)
    ax.plot(combine.index, combine['sim_old'], 's-', 
            color='red', markersize=4, label=f'Old simulation (RMSE = {rmse_old:.3f})', linewidth=1.5, alpha=0.7)
    ax.plot(combine.index, combine['sim_new'], '^-', 
            color='blue', markersize=4, label=f'New simulation (RMSE = {rmse_new:.3f})', linewidth=1.5, alpha=0.7)
    # Setup 
    ax.set_xlabel('Time')
    ax.set_ylabel('Temperature (°C)')
    ax.set_title(f'Depth {depth}m')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)
# fig.autofmt_xdate(rotation=45)
plt.tight_layout()
plt.show()

## Measured data

In [ ]:
measured_path = r'Calibration\Profiler_modem_PFL_Step.dat' # r'Calibration\Profiler_modem_SondeHourly.dat'
measured_df = pd.read_csv(measured_path, delimiter=',', header=0, skiprows=[0, 2, 3], low_memory=False)
measured_df["TIMESTAMP"] = pd.to_datetime(measured_df["TIMESTAMP"])
measured_df["temperature"] = pd.to_numeric(measured_df["sensorParms(1)"], errors='coerce')
measured_df["depth"] = pd.to_numeric(measured_df["sensorParms(9)"], errors='coerce')
measured_df = measured_df[["TIMESTAMP", "temperature", "depth"]]
# interpolate missing values in the measured data
df_interpolated = measured_df.interpolate(method='linear', limit_direction='both')
# Get data for the year 2024
measured_2024 = df_interpolated[df_interpolated["TIMESTAMP"].dt.year == 2024].reset_index(drop=True)

In [ ]:
df = (measured_2024.sort_values("TIMESTAMP").reset_index(drop=True))
new_profile = ((df["depth"] < 1.5) & (df["depth"].shift(1) > 5))
df['profile'] = new_profile.cumsum()
df

In [ ]:
depths_selected = [2, 10, 30, 50]
temp_measured = (df.groupby("profile").apply(lambda g: interpolate_profile_time(g, depths_selected))
                 .reset_index(drop=True).sort_values("TIMESTAMP").reset_index(drop=True))
temp_measured

## Read output

In [ ]:
his_path = r'Calibration\3_MonthBest\dflowfm\output\FlowFM_his.nc'
map_path = r'Calibration\3_MonthBest\dflowfm\output\FlowFM_map.nc'
his_ds = xr.open_dataset(his_path)
map_ds = xr.open_dataset(map_path)

In [ ]:
obs_name = 'Profiler'
station_names = [name.decode('utf-8').strip() for name in his_ds['station_name'].values]
# Get id of the station with the name 'Profiler'
name_id = station_names.index(obs_name) if obs_name in station_names else None
# Get the simulated temperature data for the station
temp_df = his_ds['temperature'].values[:, name_id, :]
depth_layers = map_ds['mesh2d_layer_z'].values
water_level = his_ds['waterlevel'].values[name_id][0]
depth_adjusted = water_level - depth_layers

In [ ]:
temp_interp = np.array([
    interpolate_temperature(temp_df[i], depth_adjusted, np.array(depths_selected))
    for i in range(temp_df.shape[0])
])
temp_sim = pd.DataFrame(
    temp_interp, columns=[f"sim_{depth}" for depth in depths_selected],
    index=his_ds['time'].values).reset_index().rename(columns={"index": "TIMESTAMP"}
)

In [ ]:
name = '30'
# Plot the observed and simulated temperature data
plt.figure(figsize=(12, 6))

for name in ['5', '10', '30', '40']:
    obs_col, sim_col = f'obs_{name}', f'sim_{name}'
    sim = temp_sim[["TIMESTAMP", sim_col]].dropna().reset_index(drop=True)
    obs = temp_measured[["TIMESTAMP", obs_col]].dropna().reset_index(drop=True)
    sim["TIMESTAMP"] = pd.to_datetime(sim["TIMESTAMP"])
    obs["TIMESTAMP"] = pd.to_datetime(obs["TIMESTAMP"])
    sim = sim.set_index("TIMESTAMP").sort_index()
    obs = obs.set_index("TIMESTAMP").sort_index()
    # # Remove outliers from the observed temperature data
    # obs_filled = remove_rolling_outliers(obs, obs_col)

    plt.plot(obs.index, obs[obs_col], label=f'Observed: ({name}m)')
    # plt.plot(sim.index, sim[sim_col], label=f'Sim: ({name}m)')
# plt.plot(obs_filled.index, obs_filled[obs_col], label='Observed (Outliers Removed)', color='orange')
plt.xlabel('Timestamp')
plt.ylabel('Temperature (°C)')
plt.title(f'Temperature Comparison at {obs_col}')
plt.legend()
plt.grid()
plt.show()

In [ ]:
combined = sim.reindex(sim.index.union(obs_filled.index)).sort_index()
# Interpolate the simulated temperature to match the observation timestamps
combined[sim_col] = combined[sim_col].interpolate(method="time")
result_combined = combined.loc[obs_filled.index, [sim_col]]
obs_filled.rename(columns={sim_col: obs_col}, inplace=True)
result_combined = result_combined.join(obs_filled)
result_combined = result_combined.reset_index().set_index("TIMESTAMP")
# Clip the simulated temperature to the range of observed temperature
start, end = temp_sim["TIMESTAMP"].min(), temp_sim["TIMESTAMP"].max()
result_combined = result_combined.loc[start:end]
# Compute the RMSE between simulated and observed temperature
rmse = np.sqrt(np.mean((result_combined[sim_col] - result_combined[obs_col])** 2))
# Plot the simulated and observed temperature for the specified depth
result_combined.plot(
    y=[sim_col, obs_col], figsize=(12, 6), 
    title=f"Simulated vs Observed Temperature for '{sim_col}' Depth\nRMSE: {rmse:.3f} °C", 
    ylabel="Temperature (°C)", xlabel="Time"
)
plt.show()